# Inferring Inputs: Strategy Comparison

Comparing two strategies for training with unknown feedforward inputs:
- **OU-rates**: Poisson spikes sampled from saved OU process rates (structured input)
- **Uniform-inputs**: Uniform 6 Hz Poisson spikes (unstructured input)

Both strategies optimise feedforward scaling factors (6 connections: FF→E, FF→I, E→E, E→I, I→E, I→I).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zarr
import torch
import toml

from connectome_snns.utils.reproducibility import load_experiment_config
from connectome_snns.visualization import (
    use_project_style,
    SF_PATHWAYS,
    TEACHER_COLOR,
    STRATEGY_COLORS,
    SCATTER_NEUTRAL_COLOR,
    plot_firing_rate_scatter,
    plot_r2_comparison_bar,
    plot_spike_trains,
    plot_sf_bar_chart,
)
from connectome_snns.analysis import (
    r_squared,
    fluctuation_r_squared,
    firing_rates_from_spikes,
    interleave_spike_trains,
)
from connectome_snns.analysis.inference import run_feedforward_inference
from connectome_snns.configs.conductance_based import RecurrentLayerConfig, FeedforwardLayerConfig

use_project_style()

In [ ]:
ou_rates_config = load_experiment_config("ou-rates/experiment.toml")
uniform_inputs_config = load_experiment_config("uniform-inputs/experiment.toml")

STRATEGIES = ["ou-rates", "uniform-inputs"]
STRAT_DIRS = {
    "ou-rates": ou_rates_config["output_dir"],
    "uniform-inputs": uniform_inputs_config["output_dir"],
}
PARAMS_FILES = {
    "ou-rates": ou_rates_config["parameters_file"],
    "uniform-inputs": uniform_inputs_config["parameters_file"],
}
STRAT_LABELS = {
    "ou-rates": "OU Rates",
    "uniform-inputs": "Uniform Inputs",
}
STRAT_COLORS = STRATEGY_COLORS
MAX_RATE = 40
N_NEURONS_PLOT = 10
TAU_MS = 50.0  # Gaussian kernel width for fluctuation R²
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

for strat, d in STRAT_DIRS.items():
    assert d.exists(), f"Missing: {d}"

print("Paths OK")

## Inference

Run the trained model with the best checkpoint scaling factors to obtain student spike trains for comparison with the teacher.

In [ ]:
def _load_scaling_factors_FF(state_dict, params_file):
    cfg = toml.load(params_file)
    rec_cell_params = RecurrentLayerConfig(**cfg["recurrent"]).get_cell_params()
    ff_cell_params = FeedforwardLayerConfig(**cfg["feedforward"]).get_cell_params()
    n_ff_ct = len(ff_cell_params)
    src_names = [c["name"] for c in ff_cell_params] + [
        c["name"] for c in rec_cell_params
    ]
    tgt_names = [c["name"] for c in rec_cell_params]
    src_id = {n: i for i, n in enumerate(src_names)}
    tgt_id = {n: i for i, n in enumerate(tgt_names)}
    sf = np.ones((len(src_names), len(tgt_names)), dtype=np.float32)
    for key, val in state_dict.items():
        if not key.startswith("ff_projections.") or not key.endswith(".log_sf"):
            continue
        pair = key[len("ff_projections.") : -len(".log_sf")]
        src, tgt = pair.split("__", 1)
        sf[src_id[src], tgt_id[tgt]] = float(np.exp(val.cpu().numpy()))
    return sf, n_ff_ct


def run_inference(strat_key):
    """Load best checkpoint and run inference via shared run_feedforward_inference."""
    strat_dir = STRAT_DIRS[strat_key]
    cache_path = strat_dir / "final_state" / "plot_data.npz"

    ck = torch.load(
        strat_dir / "checkpoints" / "checkpoint_best.pt",
        map_location="cpu",
        weights_only=False,
    )
    scaling_factors_FF, _ = _load_scaling_factors_FF(
        ck["model_state_dict"], PARAMS_FILES[strat_key]
    )
    print(
        f"[{strat_key}] Best checkpoint epoch={ck['epoch']}  loss={ck['best_loss']:.4f}"
    )

    ff_input = None
    if strat_key == "uniform-inputs":
        zarr_root = zarr.open_group(strat_dir / "inputs" / "spike_data.zarr", mode="r")
        dt = float(zarr_root.attrs["dt"])
        n_ff = zarr_root["input_spikes"].shape[2]
        input_spikes_stored = np.array(zarr_root["input_spikes"][0])
        avg_rate_hz = input_spikes_stored.sum() / (
            input_spikes_stored.shape[0] * dt / 1000.0 * n_ff
        )
        p_spike = avg_rate_hz * dt / 1000.0
        rng = np.random.default_rng(42)
        ff_input = (rng.random(input_spikes_stored.shape) < p_spike).astype(np.float32)
        print(f"  Uniform Poisson rate: {avg_rate_hz:.2f} Hz")

    return run_feedforward_inference(
        params_file=PARAMS_FILES[strat_key],
        run_dir=strat_dir,
        scaling_factors_FF=scaling_factors_FF,
        ff_input=ff_input,
        cache_path=cache_path,
        device=DEVICE,
        desc=strat_key,
    )


print("Inference function defined.")

In [ ]:
plot_data = {}
for strat in STRATEGIES:
    plot_data[strat] = run_inference(strat)

## Scaling Factor Recovery

In [ ]:
# ── Load final scaling factors (normalised by target) from training CSV ──────
sf_csv_keys = [key for key, _label in SF_PATHWAYS]

sf_data = {}  # strat -> dict of csv_key -> final_value
for strat, strat_dir in STRAT_DIRS.items():
    df = pd.read_csv(strat_dir / "training_metrics.csv")
    grad_rows = df[df["epoch"] >= 0]
    final = grad_rows.iloc[-1]
    sf_data[strat] = {
        k: float(final[f"scaling_factors/{k}_value"]) for k in sf_csv_keys
    }
    print(f"{strat}:")
    for k, v in sf_data[strat].items():
        print(f"  {k:<30s} {v:.4f}")

plot_sf_bar_chart(
    {STRAT_LABELS[s]: sf_data[s] for s in STRATEGIES},
    condition_labels=[STRAT_LABELS[s] for s in STRATEGIES],
    condition_colors=[STRAT_COLORS[s] for s in STRATEGIES],
    figsize=(10, 4),
    title="Final Scaling Factors vs Target",
)
plt.show()

## Firing Rate Scatter

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for col, strat in enumerate(STRATEGIES):
    d = plot_data[strat]
    dt = float(d["dt"])
    ct = d["cell_type_indices"]

    teacher_fr = firing_rates_from_spikes(d["teacher_spikes"], dt)
    student_fr = firing_rates_from_spikes(d["student_spikes"], dt)

    plot_firing_rate_scatter(
        teacher_fr,
        student_fr,
        ct,
        split_panels=True,
        axes=[axes[0, col], axes[1, col]],
        max_rate=MAX_RATE,
        marker_size=6,
        rasterized=True,
    )
    # Prefix titles with strategy name
    for row in range(2):
        title = axes[row, col].get_title()
        axes[row, col].set_title(f"{STRAT_LABELS[strat]} \u2014 {title}")

plt.tight_layout()
plt.show()

## Spike Rasters (0–5 s)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for row, strat in enumerate(STRATEGIES):
    d = plot_data[strat]
    dt = float(d["dt"])

    n_5s = int(5000.0 / dt)
    teacher = d["teacher_spikes"][:n_5s, :]
    student = d["student_spikes"][:n_5s, :]

    interleaved, ct_idx = interleave_spike_trains(
        teacher, student, n_neurons=N_NEURONS_PLOT
    )

    plot_spike_trains(
        spikes=interleaved,
        dt=dt,
        cell_type_indices=ct_idx,
        cell_type_names=["Teacher", "Student"],
        cell_type_colors={0: TEACHER_COLOR, 1: STRAT_COLORS[strat]},
        n_neurons_plot=2 * N_NEURONS_PLOT,
        n_compared=2,
        fraction=1.0,
        random_seed=None,
        title=STRAT_LABELS[strat],
        ax=axes[row],
    )

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Spike Rasters: Teacher vs Student (first 10 neurons)", fontweight="bold")
plt.tight_layout()
plt.show()

## R² Comparison

In [ ]:
r2_activity = {}
r2_fluctuation = {}

for strat in STRATEGIES:
    d = plot_data[strat]
    dt = float(d["dt"])

    teacher_fr = firing_rates_from_spikes(d["teacher_spikes"], dt)
    student_fr = firing_rates_from_spikes(d["student_spikes"], dt)

    r2_activity[strat] = r_squared(teacher_fr, student_fr)

    # Fluctuation R²
    cache_r2 = STRAT_DIRS[strat] / "final_state" / "fluct_r2.npz"
    if cache_r2.exists():
        r2_fluctuation[strat] = float(np.load(cache_r2)["r2_fluct"])
        print(f"{strat}: fluct R\u00b2={r2_fluctuation[strat]:.4f}  (cached)")
    else:
        r2_fluctuation[strat] = fluctuation_r_squared(
            d["teacher_spikes"].astype(np.float32),
            d["student_spikes"].astype(np.float32),
            TAU_MS,
            dt,
        )
        np.savez(cache_r2, r2_fluct=r2_fluctuation[strat])
        print(f"{strat}: fluct R\u00b2={r2_fluctuation[strat]:.4f}")

    print(f"{strat}: activity R\u00b2={r2_activity[strat]:.4f}")

# Bar chart
plot_r2_comparison_bar(
    [STRAT_LABELS[s] for s in STRATEGIES],
    {
        "Activity R\u00b2": [r2_activity[s] for s in STRATEGIES],
        "Fluctuation R\u00b2": [r2_fluctuation[s] for s in STRATEGIES],
    },
    condition_colors=[STRAT_COLORS[s] for s in STRATEGIES],
    title="Strategy Comparison: R\u00b2 Measures",
)
plt.show()

## Feedforward Input Comparison

Side-by-side rasters showing the OU-driven feedforward inputs vs. constant-rate Poisson inputs, plus a scatter of per-neuron firing rates across the two conditions.

In [ ]:
zarr_ou = zarr.open_group(
    STRAT_DIRS["ou-rates"] / "inputs" / "spike_data.zarr", mode="r"
)
dt_ff = float(zarr_ou.attrs["dt"])  # 1.0 ms
n_5s_ff = int(5000.0 / dt_ff)  # 5000 timesteps = 5 s
n_ff = zarr_ou["input_spikes"].shape[2]  # 1500

# ── OU latent trajectories (batch 0, 0–5 s) ──────────────────────────────────
ou_weights = np.array(zarr_ou["weights"][0, :n_5s_ff, :])  # (5000, 20)
n_patterns = ou_weights.shape[1]
time_s_ff = np.arange(n_5s_ff) * dt_ff / 1000.0

# Constant latent: uniform mixture (flat at 1/n_patterns)
const_weight = 1.0 / n_patterns

# Firing rates for scatter
ou_input_full = np.array(zarr_ou["input_spikes"][0, :, :])  # (10000, 1500)
duration_s_ff = ou_input_full.shape[0] * dt_ff / 1000.0
ou_rates = ou_input_full.sum(axis=0) / duration_s_ff
mean_ff_rate = float(ou_rates.mean())
rng_ff = np.random.default_rng(123)
p_spike = mean_ff_rate * dt_ff / 1000.0
const_input_full = rng_ff.random((int(duration_s_ff * 1000), n_ff)) < p_spike
const_rates = const_input_full.sum(axis=0) / duration_s_ff

print(f"OU mean FF rate: {mean_ff_rate:.2f} Hz  |  {n_patterns} latent dimensions")

# ── Figure 1: latent trajectories ────────────────────────────────────────────
cmap_lat = plt.cm.tab20(np.linspace(0, 1, n_patterns))

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True, sharex=True)

# Left: OU trajectories
ax = axes[0]
for k in range(n_patterns):
    ax.plot(time_s_ff, ou_weights[:, k], color=cmap_lat[k], linewidth=0.8, alpha=0.85)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Mixing Weight")
ax.set_title("OU-Process Latent Trajectories", fontweight="bold")
ax.set_xlim(0, 5)
ax.set_ylim(0, None)

# Right: constant trajectories (flat lines)
ax = axes[1]
for k in range(n_patterns):
    ax.axhline(const_weight, color=cmap_lat[k], linewidth=0.8, alpha=0.85)
ax.set_xlabel("Time (s)")
ax.set_title("Constant Latent Trajectories", fontweight="bold")
ax.set_xlim(0, 5)

fig.suptitle("Input Latent Trajectories", fontweight="bold")
plt.tight_layout()
plt.show()

# ── Figure 2: per-neuron firing rate scatter ──────────────────────────────────
ss_res = np.sum((const_rates - ou_rates) ** 2)
ss_tot = np.sum((ou_rates - ou_rates.mean()) ** 2)
r2_ff = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")

max_rate = np.percentile(ou_rates, 99) * 1.05

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(
    ou_rates, const_rates, s=4, alpha=0.4, color=SCATTER_NEUTRAL_COLOR, rasterized=True
)
ax.plot([0, max_rate], [0, max_rate], "k--", linewidth=1, alpha=0.6)
ax.set_xlim(0, max_rate)
ax.set_ylim(0, max_rate)
ax.set_aspect("equal")
ax.set_xlabel("OU-Process Rate (Hz)")
ax.set_ylabel("Constant Rate (Hz)")
ax.set_title(
    f"Input Firing Rates: OU vs Constant\n(R² = {r2_ff:.3f})", fontweight="bold"
)

plt.tight_layout()
plt.show()